## Data Loading

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("insurance.csv")

In [ ]:
df

## EDA

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
numeric_Col = ["age", "bmi", "children", "charges"]
for col in numeric_Col:
    plt.figure(figsize = (6,4))
    sns.histplot(df[col], kde = True, bins =20)

In [ ]:
sns.countplot(x = df['children'])

In [ ]:
sns.countplot(x = df['sex'])

In [ ]:
sns.countplot(x = df['smoker'])

In [ ]:
for col in numeric_Col:
    plt.figure(figsize=(6,4))
    sns.boxplot(x= df[col], color='pink')

In [ ]:
plt.Figure(figsize=(8,6))
sns.heatmap(df.corr(numeric_only=True), annot = True, linewidths=0.5)

## Data Cleaning & Preprocessing

In [ ]:
df_cleaned = df.copy()

In [ ]:
df_cleaned.drop_duplicates(inplace = True)
df_cleaned.shape

In [ ]:
df_cleaned.dtypes

In [ ]:
df_cleaned['sex'].value_counts()

In [ ]:
df_cleaned['sex'] = df_cleaned['sex'].map({'male': 0, 'female': 1})

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned['smoker'].value_counts()

In [ ]:
df_cleaned['smoker'] = df_cleaned['smoker'].map({'yes':1 , 'no':0})

In [ ]:
df_cleaned

In [ ]:
df_cleaned.rename(columns = {'sex': 'is_Female', 'smoker': 'is_smoker'}, inplace = True)

In [ ]:
df_cleaned

In [ ]:
df_cleaned['region'].value_counts()

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns = ['region'] , drop_first=True )

In [ ]:
df_cleaned

In [ ]:
df_cleaned=df_cleaned.astype(int)

In [ ]:
df_cleaned

## Feature Engineering & Extraction

In [ ]:
sns.histplot(df['bmi'], kde = True, bins =20)

In [ ]:
df_cleaned['bmi_Category'] = pd.cut(
    df_cleaned['bmi'], bins = [0, 18.5 , 24.9, 29.9, 100], labels = ['Underweight', 'Normal', 'Overweight', 'Obese']
)

In [ ]:
df_cleaned

In [ ]:
df_cleaned = pd.get_dummies(df_cleaned, columns = ['bmi_Category'], drop_first = True)


In [ ]:
df_cleaned = df_cleaned.astype(int)
df_cleaned

In [ ]:
df_cleaned.columns

In [ ]:
from sklearn.preprocessing import StandardScaler
cols = ['age', 'bmi', 'children']
scaler = StandardScaler()

df_cleaned[cols] = scaler.fit_transform(df_cleaned[cols])

In [ ]:
df_cleaned.head()

In [ ]:
df_cleaned.columns

In [ ]:
from scipy.stats import pearsonr
selected_features = ['age', 'is_Female', 'bmi', 'children', 'is_smoker',
       'region_northwest', 'region_southeast', 'region_southwest',
       'bmi_Category_Normal', 'bmi_Category_Overweight', 'bmi_Category_Obese']

correlation = {
    feature: pearsonr(df_cleaned[feature], df_cleaned['charges'])[0]
    for feature in selected_features
}

correlation_df = pd.DataFrame(list(correlation.items()), columns=['Feature', 'pearsonr Correlation'])
correlation_df.sort_values(by='pearsonr Correlation', ascending=False)

In [ ]:
categorical_features= [ 'is_Female', 'is_smoker',
       'region_northwest', 'region_southeast', 'region_southwest',
       'bmi_Category_Normal', 'bmi_Category_Overweight', 'bmi_Category_Obese']
# Chi Square Test (Category vs Category)


In [ ]:
from scipy.stats import chi2_contingency
alpha = 0.05
df_cleaned['charges_bin'] = pd.qcut(df_cleaned['charges'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
chi2_result = {}

for col in categorical_features:
    contingency = pd.crosstab(df_cleaned[col], df_cleaned['charges_bin'])
    chi2_stat, p_val, _, _ = chi2_contingency(contingency)
    decision= "Reject Null Hypothesis" if p_val < alpha else "Accept Null Hypothesis"
    chi2_result[col] = {
        'chi2_statistics' : chi2_stat,
        'p_value' : p_val,
        'decision' : decision
    }

In [ ]:
chi2_df = pd.DataFrame(chi2_result).T
chi2_df = chi2_df.sort_values(by='p_value', ascending=True)
chi2_df

In [ ]:
final_df = df_cleaned[['age', 'is_Female', 'bmi', 'children', 'is_smoker', 'bmi_Category_Normal', 'bmi_Category_Overweight', 'bmi_Category_Obese', 'charges']]
final_df